# Preprocessing and Harmonization

Clean, harmonize, and merge raw datasets into an analytical dataset.

## Setup

In [ ]:
import sys
import pathlib
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

project_root = pathlib.Path().resolve().parent
sys.path.insert(0, str(project_root))

from src.preprocessing import load_ef_epi, clean_eurostat, compute_percentiles, merge_datasets

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Load Raw Data

In [ ]:
import eurostat

eurostat_raw = eurostat.get_data_df('educ_uoe_lang01', flags=False)
ef_raw = pd.read_csv(project_root / 'data' / 'raw' / 'efiepi_rankings.csv')

print(f'Eurostat: {eurostat_raw.shape}')
print(f'EF: {ef_raw.shape}')

## Clean Eurostat Data

Filter for English language, total sex, percentage units, and national-level (2-letter country codes).

In [ ]:
eurostat_df = clean_eurostat(eurostat_raw)
print(f'Cleaned: {eurostat_df.shape}')
print(f'Columns: {list(eurostat_df.columns)}')
print(f'Countries: {eurostat_df["iso3"].nunique()}')
print(f'Year range: {int(eurostat_df["year"].min())} to {int(eurostat_df["year"].max())}')
print()
eurostat_df.head()

## Load and Clean EF Data

Harmonize country names to ISO 3166-1 alpha-3 codes.

In [ ]:
ef_df = load_ef_epi(str(project_root / 'data' / 'raw' / 'efiepi_rankings.csv'))
print(f'Cleaned: {ef_df.shape}')
print(f'Columns: {list(ef_df.columns)}')
print(f'Countries: {ef_df["iso3"].nunique()}')
print(f'Year range: {int(ef_df["year"].min())} to {int(ef_df["year"].max())}')
print()
ef_df.head()

## Compute Percentiles

Within each year, compute percentile ranks of learning exposure and EF proficiency to enable cross-year comparison.

In [ ]:
eurostat_df = compute_percentiles(eurostat_df, 'learning', ['year'], 'learning_percentile')
ef_df = compute_percentiles(ef_df, 'score', ['year'], 'ef_percentile')

print('Percentiles computed')
print()
print('Eurostat percentile range:', eurostat_df['learning_percentile'].min(), 'to', eurostat_df['learning_percentile'].max())
print('EF percentile range:', ef_df['ef_percentile'].min(), 'to', ef_df['ef_percentile'].max())

## Merge Datasets with Lag Alignment

Align EF proficiency (year t+4) with ESL exposure (year t) to reflect policy timing and skill development.

In [ ]:
merged = merge_datasets(eurostat_df, ef_df, lag_years=4)

print(f'Merged shape: {merged.shape}')
print(f'Columns: {list(merged.columns)}')
print(f'Countries: {merged["iso3"].nunique()}')
print(f'Year range: {int(merged["year"].min())} to {int(merged["year"].max())}')
print()
merged.head(10)

## Gap Variable

gap_pct = EF_percentile − Learning_percentile  
Positive gap: high proficiency relative to learning exposure. Negative gap: low proficiency relative to exposure.

In [ ]:
print('Gap statistics:')
print(merged['gap_pct'].describe())
print()
print('Non-null gap values:', merged['gap_pct'].notna().sum())

## Validation

In [ ]:
print('Validation:')
print(f'  Duplicates (iso3-year): {merged.duplicated(subset=["iso3", "year"]).sum()}')
print(f'  Missing values: {merged.isnull().sum().sum()}')
print(f'  Complete cases: {merged.dropna().shape[0]} / {merged.shape[0]} ({merged.dropna().shape[0]/merged.shape[0]*100:.1f}%)')
print(f'  EF match rate: {merged["ef_percentile"].notna().sum() / merged.shape[0] * 100:.1f}%')

## Export Dataset

In [ ]:
output_path = project_root / 'data' / 'processed' / 'merged_analytical.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)

merged_sorted = merged.sort_values(['year', 'iso3']).reset_index(drop=True)
merged_sorted.to_csv(output_path, index=False)

print(f'Exported to: {output_path}')
print(f'  Rows: {len(merged_sorted)}')
print(f'  Columns: {len(merged_sorted.columns)}')
print(f'  Size: {output_path.stat().st_size / 1024:.1f} KB')